# Deep Learning History, CNN, YOLO and Image Processing
A simple, beginner-friendly notebook.

**Order of this notebook**
1. Deep learning history (start to now)
2. CNN: how many conv layers in famous models
3. YOLO: how it is different from conv layers, and its new idea
4. Image processing concepts, step by step, with code
5. Link: image processing -> CNN -> YOLO

Run each code cell from top to bottom. You only need `numpy` and `matplotlib`.

## 1. Deep Learning History (Start to End)

| Year | Event | Why it matters (simple) |
|---|---|---|
| 1943 | McCulloch and Pitts: first artificial neuron | First math idea of a brain cell |
| 1958 | Rosenblatt: **Perceptron** | First neuron that can learn |
| 1969 | Minsky and Papert show perceptron limits | Funding drops: **first AI winter** |
| 1986 | Rumelhart, Hinton, Williams: **Backpropagation** | Way to train many layers |
| 1989 | Yann LeCun: first CNN for handwritten digits | Start of CNNs |
| 1998 | **LeNet-5** | CNN reads bank cheques and zip codes |
| 1997 | **LSTM** (Hochreiter, Schmidhuber) | Memory for sequences |
| 2006 | Hinton: deep belief networks | Name "deep learning" becomes popular |
| 2009 | **ImageNet** dataset (Fei-Fei Li) | Big data for images |
| 2012 | **AlexNet** wins ImageNet by a big gap | The deep learning **revolution** (GPUs + big data) |
| 2014 | **GAN** (Goodfellow), VGG, GoogLeNet | Image generation, deeper CNNs |
| 2015 | **ResNet** (152 layers), **YOLO**, Faster R-CNN | Very deep nets, fast object detection |
| 2017 | **Transformer** ("Attention is all you need") | New base for language models |
| 2018 | BERT, GPT | Pretrained language models |
| 2020 | **Vision Transformer (ViT)**, GPT-3 | Transformers for images, very large language models |
| 2021-22 | CLIP, **Diffusion models**, ChatGPT | Text-to-image and chat AI for everyone |
| 2023-now | Large multimodal models and agents | Text, image, audio in one model |

### The 3 big reasons for the revolution (around 2012)
1. **Data**: ImageNet gave millions of labeled images.
2. **Compute**: GPUs made training fast.
3. **Better methods**: ReLU, dropout, better initialization, batch normalization.

## 2. CNN: Layers and How Many Conv Layers

A CNN has these main layers:
- **Convolution (conv) layer**: slides small filters over the image to find patterns (edges, shapes).
- **Activation (ReLU)**: keeps positive values, removes negative values.
- **Pooling**: makes the image smaller (keeps the strongest value).
- **Fully connected (dense) layer**: makes the final decision.

### Number of conv layers in famous models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = {
    "LeNet-5 (1998)": 2,     # 2 conv + 1 conv-like C5 layer, often counted as 2-3
    "AlexNet (2012)": 5,
    "VGG-16 (2014)": 13,
    "VGG-19 (2014)": 16,
    "ResNet-50 (2015)": 49,  # 49 conv + 1 fully connected = 50 layers
    "YOLOv1 (2015)": 24,     # 24 conv + 2 fully connected
}

for name, n in models.items():
    print(f"{name:20s} -> {n} conv layers")

plt.figure(figsize=(8, 4))
plt.barh(list(models.keys()), list(models.values()), color="steelblue")
plt.xlabel("Number of conv layers")
plt.title("Conv layers in famous models")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

Note: "How many layers" can change with how people count (some count pooling or dense layers). The numbers above are the common values from the original papers.

**Trend:** more layers = the network can learn more complex patterns. ResNet added *skip connections* so very deep networks can be trained.

## 3. YOLO: What is Different from Conv Layers?

**Important:** YOLO is *not* a layer. A **conv layer is a building block**. **YOLO is a full model** (a way to do object detection) that is *built using* conv layers.

| | Conv layer | YOLO |
|---|---|---|
| What is it? | One layer (a building block) | A complete detection model |
| Job | Find features (edges, shapes) | Find **what** objects are and **where** they are |
| Output | Feature map | Boxes + class names + confidence |

### What was before YOLO?
**R-CNN family (2014)**: first find ~2000 possible regions, then run a CNN on each region, then classify. It was **slow** (many steps).

### YOLO's new technique (2015, Redmon et al.): "You Only Look Once"
1. **One pass**: the whole image goes through one network one time.
2. **Detection as regression**: the network directly predicts boxes and classes (no separate region step).
3. **Grid idea**: divide the image into an `S x S` grid. The cell that holds the **center** of an object is responsible for it.
4. Each cell predicts `B` boxes (x, y, w, h, confidence) and class probabilities.
5. **Output shape (YOLOv1):** `7 x 7 x (2*5 + 20) = 7 x 7 x 30`.
6. Clean up with **IoU** and **NMS** (remove duplicate boxes).

Result: very fast, real-time detection.

### YOLO versions (short)
v1 (2015) -> v2 (anchor boxes) -> v3 (3 scales) -> v4/v5 (better training tricks) -> v7, v8, v9, v10, v11 (faster, more accurate).

In [ ]:
# YOLO idea 1: Grid. Which cell is responsible for an object?
S = 7                      # 7 x 7 grid
img_w, img_h = 448, 448    # YOLOv1 input size

# One object: box center and size (in pixels)
cx, cy, w, h = 300, 150, 120, 160

cell_w, cell_h = img_w / S, img_h / S
col = int(cx // cell_w)
row = int(cy // cell_h)
print(f"Object center is in grid cell: row={row}, col={col}")

# Output size of YOLOv1
B, C = 2, 20
print("YOLOv1 output shape:", (S, S, B * 5 + C))

In [ ]:
# YOLO idea 2: IoU (Intersection over Union) = how much two boxes overlap
def iou(a, b):
    # box format: (x1, y1, x2, y2)
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-9)

print("IoU same box     :", round(iou((0,0,10,10), (0,0,10,10)), 2))
print("IoU half overlap :", round(iou((0,0,10,10), (5,0,15,10)), 2))
print("IoU no overlap   :", round(iou((0,0,10,10), (20,20,30,30)), 2))

In [ ]:
# YOLO idea 3: NMS (Non-Maximum Suppression) = remove duplicate boxes
def nms(boxes, scores, iou_thresh=0.5):
    order = np.argsort(scores)[::-1]       # best score first
    keep = []
    while len(order) > 0:
        best = order[0]
        keep.append(best)
        rest = order[1:]
        order = np.array([i for i in rest if iou(boxes[best], boxes[i]) < iou_thresh])
    return keep

boxes = [(10,10,60,60), (12,12,62,62), (100,100,150,150)]
scores = [0.9, 0.8, 0.7]
keep = nms(boxes, scores)
print("Boxes kept:", keep, "-> box 1 was a duplicate of box 0 and was removed")

## 4. Image Processing Concepts (in Order)

Image processing = changing or reading an image using maths, **before or without** deep learning.
We will use a small **fake image** (so no download is needed).

Order:
1. Image = numbers (pixels)
2. Grayscale
3. Brightness and contrast
4. Histogram
5. Thresholding
6. Convolution (filter) and blur
7. Sharpen
8. Edge detection (Sobel)
9. Morphology (erode, dilate)
10. Resize, rotate, flip
11. Pooling

### Step 1: Image = numbers (pixels)

In [ ]:
def make_image(seed=0):
    rng = np.random.default_rng(seed)
    img = np.tile(np.linspace(0.1, 0.4, 64), (64, 1))      # soft background
    img[10:30, 10:30] = 0.85                               # bright square
    yy, xx = np.ogrid[:64, :64]
    circle = (yy - 45) ** 2 + (xx - 45) ** 2 <= 10 ** 2    # circle
    img[circle] = 0.65
    img += rng.normal(0, 0.03, img.shape)                  # small noise
    return np.clip(img, 0, 1)

img = make_image()
print("Shape:", img.shape, "| min:", img.min().round(2), "| max:", img.max().round(2))
print("Top-left 3x3 pixel values:\n", img[:3, :3].round(2))

def show(images, titles, cmap="gray"):
    fig, axes = plt.subplots(1, len(images), figsize=(3.2 * len(images), 3.4))
    axes = np.atleast_1d(axes)
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im, cmap=cmap, vmin=0, vmax=1 if im.max() <= 1 else None)
        ax.set_title(t, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show([img], ["Original (grayscale numbers 0 to 1)"])

### Step 2: Grayscale
A colour image has 3 channels (R, G, B). Grayscale mixes them into 1 channel.
Formula: `gray = 0.299*R + 0.587*G + 0.114*B`

In [ ]:
rgb = np.stack([img, img * 0.7, img * 0.4], axis=-1)   # make a fake colour image
gray = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
print("Colour shape:", rgb.shape, "-> Gray shape:", gray.shape)
fig, ax = plt.subplots(1, 2, figsize=(6.5, 3.4))
ax[0].imshow(np.clip(rgb, 0, 1)); ax[0].set_title("Colour (3 channels)"); ax[0].axis("off")
ax[1].imshow(gray, cmap="gray"); ax[1].set_title("Gray (1 channel)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

### Step 3: Brightness and contrast
- Brightness: add a number to every pixel.
- Contrast: multiply every pixel by a number.

In [ ]:
bright = np.clip(img + 0.25, 0, 1)
contrast = np.clip((img - 0.5) * 1.8 + 0.5, 0, 1)
show([img, bright, contrast], ["Original", "Brighter (+0.25)", "More contrast (x1.8)"])

### Step 4: Histogram
A histogram counts how many pixels have each brightness. **Histogram equalization** spreads brightness values to improve contrast.

In [ ]:
def equalize(im):
    levels = (im * 255).astype(int)
    hist = np.bincount(levels.ravel(), minlength=256)
    cdf = hist.cumsum()
    cdf = (cdf - cdf.min()) / (cdf.max() - cdf.min())
    return cdf[levels]

eq = equalize(img)
fig, ax = plt.subplots(2, 2, figsize=(7, 6))
ax[0, 0].imshow(img, cmap="gray"); ax[0, 0].set_title("Original"); ax[0, 0].axis("off")
ax[0, 1].hist(img.ravel(), bins=50, color="gray"); ax[0, 1].set_title("Histogram (original)")
ax[1, 0].imshow(eq, cmap="gray"); ax[1, 0].set_title("Equalized"); ax[1, 0].axis("off")
ax[1, 1].hist(eq.ravel(), bins=50, color="gray"); ax[1, 1].set_title("Histogram (equalized)")
plt.tight_layout(); plt.show()

### Step 5: Thresholding
Pixel above the limit -> white (1). Below -> black (0). Used to separate object from background.

In [ ]:
binary = (img > 0.55).astype(float)
show([img, binary], ["Original", "Threshold > 0.55"])

### Step 6: Convolution and blur
**Convolution** = slide a small grid (kernel/filter) over the image, multiply and add.
This is exactly what a **conv layer** does in a CNN. The difference: in image processing we **choose** the kernel by hand; in a CNN the network **learns** the kernel.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def conv2d(im, kernel):
    k = kernel.shape[0]
    padded = np.pad(im, k // 2, mode="edge")
    windows = sliding_window_view(padded, (k, k))
    return (windows * kernel).sum(axis=(-1, -2))

blur_kernel = np.ones((5, 5)) / 25          # average of 25 pixels
blurred = conv2d(img, blur_kernel)
show([img, blurred], ["Original (noisy)", "Blur 5x5 (less noise)"])

### Step 7: Sharpen

In [ ]:
sharpen_kernel = np.array([[ 0, -1,  0],
                           [-1,  5, -1],
                           [ 0, -1,  0]])
sharp = np.clip(conv2d(blurred, sharpen_kernel), 0, 1)
show([blurred, sharp], ["Blurred", "Sharpened"])

### Step 8: Edge detection (Sobel)
An edge = a place where brightness changes fast. Sobel finds change in x (vertical edges) and y (horizontal edges).
CNN first layers often learn filters that look like these.

In [ ]:
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])
sobel_y = sobel_x.T

gx = conv2d(blurred, sobel_x)
gy = conv2d(blurred, sobel_y)
edges = np.sqrt(gx ** 2 + gy ** 2)
edges = edges / edges.max()
show([blurred, np.abs(gx) / np.abs(gx).max(), np.abs(gy) / np.abs(gy).max(), edges],
     ["Blurred", "Edges in x", "Edges in y", "Edge strength"])

### Step 9: Morphology (erode and dilate)
Works on black/white images.
- **Erode**: shrinks white areas (removes small white dots).
- **Dilate**: grows white areas (fills small holes).

In [ ]:
def erode(b, k=3):
    padded = np.pad(b, k // 2, mode="constant")
    return sliding_window_view(padded, (k, k)).min(axis=(-1, -2))

def dilate(b, k=3):
    padded = np.pad(b, k // 2, mode="constant")
    return sliding_window_view(padded, (k, k)).max(axis=(-1, -2))

show([binary, erode(binary), dilate(binary)], ["Binary", "Eroded", "Dilated"])

### Step 10: Resize, rotate, flip

In [ ]:
def resize_nearest(im, new_h, new_w):
    rows = (np.arange(new_h) * im.shape[0] / new_h).astype(int)
    cols = (np.arange(new_w) * im.shape[1] / new_w).astype(int)
    return im[rows][:, cols]

small = resize_nearest(img, 32, 32)
show([img, small, np.rot90(img), np.fliplr(img)],
     ["Original 64x64", "Resized 32x32", "Rotated 90", "Flipped"])

### Step 11: Pooling
Max pooling keeps the largest value in each block. It makes the image smaller but keeps strong features. CNNs use it between conv layers.

In [ ]:
def max_pool(im, size=2):
    h, w = im.shape
    h, w = h - h % size, w - w % size
    blocks = im[:h, :w].reshape(h // size, size, w // size, size)
    return blocks.max(axis=(1, 3))

pooled = max_pool(edges, 2)
print("Before pooling:", edges.shape, "-> After pooling:", pooled.shape)
show([edges, pooled], ["Edge map 64x64", "After max pool 32x32"])

## 5. Link: Image Processing -> CNN -> YOLO

| Image processing step | Where it appears in deep learning |
|---|---|
| Pixels as numbers | Input of every CNN |
| Convolution with hand-made kernel | **Conv layer** (kernel is **learned**) |
| Edge filters (Sobel) | Early conv layers learn edge-like filters |
| Pooling | Pooling layers (reduce size) |
| Resize | YOLO resizes input to a fixed size (for example 448x448 or 640x640) |
| IoU, NMS | Used in YOLO to score and clean boxes |

**Full path:** pixels -> conv layers find edges -> deeper layers find shapes and objects -> YOLO head predicts boxes and classes in one pass.

### Next steps for a data analyst beginner
1. Run this notebook and change the kernel values to see what happens.
2. Try `OpenCV` (`cv2`) and `Pillow` on a real photo.
3. Build a small CNN with Keras on the MNIST dataset.
4. Try a pretrained YOLO model (for example with the `ultralytics` package).